In [ ]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.3 MB/s eta 0:00:00


Connect to the folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Define the path to the 'Pest Detection' folder
folder_path = '/content/drive/MyDrive/Pest Detection'

# Check if the folder exists and then change the directory
if os.path.exists(folder_path):
    os.chdir(folder_path)
    print(f"Successfully changed directory to: {os.getcwd()}")
else:
    print(f"The folder '{folder_path}' does not exist. Please ensure it is correctly named and located in your Google Drive.")
    print("Current working directory is still:", os.getcwd())

# Verify the current working directory
%pwd

Successfully changed directory to: /content/drive/MyDrive/Pest Detection


'/content/drive/MyDrive/Pest Detection'

## Create YOLO Directory Structure

In [ ]:
import os

base_dir = 'dataset'

# Create the base dataset directory
os.makedirs(base_dir, exist_ok=True)
print(f"Created directory: {base_dir}")

# Define subdirectories for train, val, test
subdirs = ['train', 'val', 'test']

for subdir in subdirs:
    images_path = os.path.join(base_dir, subdir, 'images')
    labels_path = os.path.join(base_dir, subdir, 'labels')

    os.makedirs(images_path, exist_ok=True)
    os.makedirs(labels_path, exist_ok=True)

    print(f"Created directory: {images_path}")
    print(f"Created directory: {labels_path}")

print("YOLO directory structure created successfully.")

Created directory: dataset
Created directory: dataset/train/images
Created directory: dataset/train/labels
Created directory: dataset/val/images
Created directory: dataset/val/labels
Created directory: dataset/test/images
Created directory: dataset/test/labels
YOLO directory structure created successfully.


## Process and Split Data

Iterate through the specified number ranges (1-250, 251-500, 501-750, 751-1000). For each range, gather image and label files, then split them into 80% training, 10% validation, and 10% testing sets. Copy the files to their respective directories within the YOLO structure.


In [ ]:
import shutil
import random

# Define paths for raw images and labels
raw_images_dir = 'Image'
raw_labels_dir = 'Label'

# --- Diagnostic Check Start ---
print(f"Current working directory: {os.getcwd()}")

if not os.path.exists(raw_images_dir):
    print(f"Error: Raw images directory '{raw_images_dir}' not found at {os.path.join(os.getcwd(), raw_images_dir)}")
else:
    print(f"Raw images directory '{raw_images_dir}' found. Contains: {os.listdir(raw_images_dir)[:5]}...")

if not os.path.exists(raw_labels_dir):
    print(f"Error: Raw labels directory '{raw_labels_dir}' not found at {os.path.join(os.getcwd(), raw_labels_dir)}")
else:
    print(f"Raw labels directory '{raw_labels_dir}' found. Contains: {os.listdir(raw_labels_dir)[:5]}...")
print("-- - Diagnostic Check End ---")

# List to store base filenames (e.g., '001')
available_files = []

# Iterate through the specified number ranges
for start, end in [(1, 250), (251, 500), (501, 750), (751, 1000)]:
    for i in range(start, end + 1):
        # Format the file number with leading zeros (e.g., 001, 010, 100)
        file_num = str(i)
        image_filename = f'{file_num}.jpg'
        label_filename = f'{file_num}.txt'

        image_path = os.path.join(raw_images_dir, image_filename)
        label_path = os.path.join(raw_labels_dir, label_filename)

        # Check if both image and label files exist
        if os.path.exists(image_path) and os.path.exists(label_path):
            available_files.append(file_num)
        # else:
        #     # Uncomment for detailed debugging of missing files
        #     # if not os.path.exists(image_path):
        #     #     print(f"Missing image: {image_path}")
        #     # if not os.path.exists(label_path):
        #     #     print(f"Missing label: {label_path}")

print(f"Found {len(available_files)} image-label pairs.")

# Shuffle the list of available filenames
random.shuffle(available_files)

# Calculate split points
total_files = len(available_files)
train_split = int(0.8 * total_files)
val_split = int(0.1 * total_files)

# Divide files into train, val, test sets
train_files = available_files[:train_split]
val_files = available_files[train_split : train_split + val_split]
test_files = available_files[train_split + val_split :]

print(f"Train files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")
print(f"Test files: {len(test_files)}")

# Function to copy files
def copy_files(file_list, image_dest_dir, label_dest_dir):
    for file_num in file_list:
        # Original paths
        src_image_path = os.path.join(raw_images_dir, f'{file_num}.jpg')
        src_label_path = os.path.join(raw_labels_dir, f'{file_num}.txt')

        # Destination paths
        dest_image_path = os.path.join(image_dest_dir, f'{file_num}.jpg')
        dest_label_path = os.path.join(label_dest_dir, f'{file_num}.txt')

        # Copy files
        shutil.copy(src_image_path, dest_image_path)
        shutil.copy(src_label_path, dest_label_path)

# Copy files to their respective directories
copy_files(train_files, os.path.join(base_dir, 'train', 'images'), os.path.join(base_dir, 'train', 'labels'))
copy_files(val_files, os.path.join(base_dir, 'val', 'images'), os.path.join(base_dir, 'val', 'labels'))
copy_files(test_files, os.path.join(base_dir, 'test', 'images'), os.path.join(base_dir, 'test', 'labels'))

print("Files successfully copied to train, val, and test directories.")

Current working directory: /content/drive/MyDrive/Pest Detection
Raw images directory 'Image' found. Contains: ['983.jpg', '978.jpg', '999.jpg', '929.jpg', '973.jpg']...
Raw labels directory 'Label' found. Contains: ['139.txt', '240.txt', '113.txt', '56.txt', '54.txt']...
-- - Diagnostic Check End ---
Found 1000 image-label pairs.
Train files: 800
Validation files: 100
Test files: 100
Files successfully copied to train, val, and test directories.


In [ ]:
pip install pyyaml

In [ ]:
import yaml
import os

# Get the absolute path to the dataset directory
absolute_dataset_path = os.path.join(os.getcwd(), base_dir)

# Define the content for data.yaml
data_yaml_content = {
    'path': absolute_dataset_path, # Absolute path to the dataset folder
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 3, # Number of classes
    'names': ['Bo_den', 'Bo_den_thon', 'Sau_duc_than'] # Class names
}

# Define the path where data.yaml will be saved
data_yaml_path = os.path.join(base_dir, 'data.yaml')

# Write the dictionary to a YAML file
with open(data_yaml_path, 'w') as file:
    yaml.dump(data_yaml_content, file, default_flow_style=False)

print(f"'{data_yaml_path}' created successfully with the following content:")
with open(data_yaml_path, 'r') as file:
    print(file.read())

'dataset/data.yaml' created successfully with the following content:
names:
- Bo_den
- Bo_den_thon
- Sau_duc_than
nc: 3
path: /content/drive/MyDrive/Pest Detection/dataset
test: test/images
train: train/images
val: val/images



In [ ]:
!ls

dataset			   results_image38.jpg	results_image70.jpg
data.yaml		   results_image39.jpg	results_image71.jpg
Image			   results_image3.jpg	results_image72.jpg
Label			   results_image40.jpg	results_image73.jpg
pest_detection_yolov8m.pt  results_image41.jpg	results_image74.jpg
results_image0.jpg	   results_image42.jpg	results_image75.jpg
results_image10.jpg	   results_image43.jpg	results_image76.jpg
results_image11.jpg	   results_image44.jpg	results_image77.jpg
results_image12.jpg	   results_image45.jpg	results_image78.jpg
results_image13.jpg	   results_image46.jpg	results_image79.jpg
results_image14.jpg	   results_image47.jpg	results_image7.jpg
results_image15.jpg	   results_image48.jpg	results_image80.jpg
results_image16.jpg	   results_image49.jpg	results_image81.jpg
results_image17.jpg	   results_image4.jpg	results_image82.jpg
results_image18.jpg	   results_image50.jpg	results_image83.jpg
results_image19.jpg	   results_image51.jpg	results_image84.jpg
results_image1.jpg	   results_image

# Task
Train a YOLOv8m model for pest detection using the `data.yaml` dataset, leveraging available GPUs for multi-GPU training over 50 epochs with an image size of 640 and a batch size of 16.

## Load YOLO Model

Import the `YOLO` class and load the `yolov8m.pt` pre-trained model.


In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8m model
model = YOLO('yolov8m.pt')

# Step 5: Train the model
results = model.train(
    data="data.yaml",     # path to your dataset yaml
    epochs=50,            # number of epochs
    imgsz=640,            # image size
    batch=16,             # batch size
    name="result",    # folder name for results
    device = 0
)

# Step 6: Evaluate the model
metrics = model.val()

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=result, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

In [ ]:
model.save("pest_detection_yolov8m.pt")  # You can give any filename you like

# Test Model

In [ ]:
from ultralytics import YOLO
import glob

test_images = glob.glob("dataset/test/images/*.jpg")
results = model.predict(test_images, conf=0.25)  # adjust confidence as needed
for r in results:
    r.save()  # save each result separately


0: 480x640 11 Bo_dens, 5 Bo_den_thons, 21.0ms
1: 480x640 6 Bo_dens, 1 Bo_den_thon, 21.0ms
2: 480x640 3 Bo_dens, 21.0ms
3: 480x640 12 Bo_dens, 2 Bo_den_thons, 21.0ms
4: 480x640 2 Bo_dens, 2 Bo_den_thons, 21.0ms
5: 480x640 (no detections), 21.0ms
6: 480x640 21 Bo_dens, 1 Bo_den_thon, 21.0ms
7: 480x640 8 Bo_dens, 5 Bo_den_thons, 21.0ms
8: 480x640 17 Bo_dens, 4 Bo_den_thons, 21.0ms
9: 480x640 23 Bo_dens, 1 Bo_den_thon, 21.0ms
10: 480x640 20 Bo_dens, 1 Bo_den_thon, 21.0ms
11: 480x640 4 Bo_dens, 1 Bo_den_thon, 21.0ms
12: 480x640 5 Bo_dens, 21.0ms
13: 480x640 2 Bo_dens, 21.0ms
14: 480x640 33 Bo_dens, 3 Bo_den_thons, 21.0ms
15: 480x640 1 Bo_den, 1 Sau_duc_than, 21.0ms
16: 480x640 5 Bo_dens, 3 Bo_den_thons, 21.0ms
17: 480x640 5 Bo_dens, 3 Bo_den_thons, 21.0ms
18: 480x640 1 Bo_den, 1 Bo_den_thon, 21.0ms
19: 480x640 7 Bo_dens, 3 Bo_den_thons, 21.0ms
20: 480x640 13 Bo_dens, 14 Bo_den_thons, 1 Sau_duc_than, 21.0ms
21: 480x640 3 Bo_dens, 21.0ms
22: 480x640 13 Bo_dens, 1 Bo_den_thon, 21.0ms
23: 480x

In [ ]:
results = model.val(data="data.yaml", split="test", name="test" )

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 7.2±14.7 ms, read: 27.8±23.3 MB/s, size: 83.7 KB)
val: Scanning /content/drive/MyDrive/Pest Detection/dataset/test/labels.cache... 100 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 26.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.2it/s 5.6s
                   all        100       1449      0.672      0.676      0.686      0.369
                Bo_den         91        972      0.696      0.865      0.811      0.417
           Bo_den_thon         80        457       0.64      0.464      0.565      0.253
          Sau_duc_than         17         20      0.681        0.7      0.682      0.438
Speed: 6.5ms preprocess, 20.2ms inference, 0.0ms loss, 6.8ms postprocess per image
Results saved to /content/drive/MyDrive/Pest Detection/runs/detect/test


# Load Best Train Model

In [ ]:
import os
from ultralytics import YOLO

filepath = "runs/detect/result/weights/best.pt"

if os.path.exists(filepath):
    print("File exists")
else:
    print("File does not exist")

model = YOLO(filepath)

results = model.val(data="data.yaml", split="test", name="best-test" )

File exists
Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.4 ms, read: 1.7±3.3 MB/s, size: 97.0 KB)
val: Scanning /content/drive/MyDrive/Pest Detection/dataset/test/labels.cache... 100 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 18.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.4it/s 5.2s
                   all        100       1449      0.672      0.676      0.686      0.369
                Bo_den         91        972      0.696      0.865      0.811      0.417
           Bo_den_thon         80        457       0.64      0.464      0.565      0.253
          Sau_duc_than         17         20      0.681        0.7      0.682      0.438
Speed: 3.2ms preprocess, 17.2ms inference, 0.0ms loss, 12.3ms postprocess per image
Results saved to /c

# Statistic

||mAP50|mAP50-95|
|---|---|---|
|Train|0.716|0.379|
|Val|0.719|0.383|
|test|0.686|0.369|